# Experiment: TravelPlanner Official Full Evaluation

This notebook runs the complete TravelPlanner official validation campaign through Docker, stores one structured artifact per query, aggregates exported `final_plan` objects, and launches the official scorer on the full run set.

The benchmark path stays containerized end-to-end. The notebook is only the driver.


## Workflow

1. Build or refresh the `travelplanner-smoke` Docker image.
2. Ensure the TravelPlanner database is present under `data/travelplanner/database`.
3. Execute the selected split query by query with `scripts/run_travelplanner_query_export.py`.
4. Resume safely from `runs.json` if the campaign is interrupted.
5. Run `scripts/eval_travelplanner_official.py` on the aggregated outputs.
6. Inspect official scores, total cost, total tokens, and runtime failures.

## Notes

- `MAX_QUERIES = None` means the full split.
- If a query run fails, the notebook records an empty plan for that query so the official evaluation still uses the full denominator.
- Docker must be running and `.env` must contain `OPENROUTER_API_KEY`.


In [7]:
import json
import os
import shlex
import subprocess
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "docker-compose.yml").exists() and (candidate / "main.py").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from the current working directory.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
print(f"REPO_ROOT={REPO_ROOT}")

def repo_rel(path: Path) -> str:
    return path.resolve().relative_to(REPO_ROOT).as_posix()

def run_command(cmd, *, cwd: Path = REPO_ROOT, env: dict | None = None, log_path: Path | None = None, check: bool = False):
    merged_env = os.environ.copy()
    if env:
        merged_env.update({key: str(value) for key, value in env.items()})
    rendered = shlex.join([str(part) for part in cmd])
    print(f"$ {rendered}")
    proc = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=str(cwd),
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    output_lines: list[str] = []
    for line in proc.stdout:
        print(line, end="")
        output_lines.append(line)
    returncode = proc.wait()
    output = "".join(output_lines)
    if log_path is not None:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text(output, encoding="utf-8")
    if check and returncode != 0:
        raise RuntimeError(f"Command failed with exit={returncode}: {rendered}")
    return {"returncode": returncode, "output": output, "log_path": str(log_path) if log_path is not None else ""}

def extract_tail_json(text: str) -> dict:
    lines_ = text.splitlines()
    for index in range(len(lines_) - 1, -1, -1):
        if not lines_[index].lstrip().startswith("{"):
            continue
        candidate = "\n".join(lines_[index:]).strip()
        if not candidate:
            continue
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            continue
    raise ValueError("Could not extract a trailing JSON object from command output.")

def docker_compose_run(*command: str, service: str = "travelplanner-smoke", env_vars: dict | None = None) -> list[str]:
    cmd = ["docker", "compose", "run", "--rm"]
    for key, value in (env_vars or {}).items():
        cmd.extend(["-e", f"{key}={value}"])
    cmd.append(service)
    cmd.extend(command)
    return cmd

def save_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

def load_json(path: Path, default):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))

def synthetic_failed_run(query_idx: int, *, result: dict, error: str) -> dict:
    return {
        "query_idx": int(query_idx),
        "status": "runtime_error",
        "error": error,
        "returncode": int(result.get("returncode", 1)),
        "final_plan": [],
        "plan": [],
        "summary": None,
        "log_path": str(result.get("log_path", "")),
    }


REPO_ROOT=/Users/lotfi/Documents/EMLV/Memoire/StigmergiAgentic


In [8]:
RUN_TAG = os.environ.get("TRAVELPLANNER_FULL_EVAL_RUN_TAG") or datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
SPLIT = "validation"
START_QUERY_IDX = 0
MAX_QUERIES = None  # None => full split
SEED = 42
MAX_TICKS = 30
CONTINUE_ON_ERROR = True
BUILD_IMAGE = True
PREPARE_DATA = True

EVAL_ROOT = REPO_ROOT / "output" / "travelplanner_official_full_eval"
RUN_DIR = EVAL_ROOT / RUN_TAG
QUERY_DIR = RUN_DIR / "queries"
HF_CACHE_DIR = RUN_DIR / "hf_cache"
RUNS_JSON = RUN_DIR / "runs.json"
OFFICIAL_JSON = RUN_DIR / "official_eval.json"
OFFICIAL_LOG = RUN_DIR / "official_eval.log"
BUILD_LOG = RUN_DIR / "docker_build.log"
DATA_LOG = RUN_DIR / "setup_data.log"
COUNT_LOG = RUN_DIR / "dataset_count.log"
QUERY_RUNNER = REPO_ROOT / "scripts" / "run_travelplanner_query_export.py"

for path in (RUN_DIR, QUERY_DIR, HF_CACHE_DIR):
    path.mkdir(parents=True, exist_ok=True)

DOCKER_ENV = {
    "PYTHONUNBUFFERED": "1",
    "HF_HOME": f"/app/{repo_rel(HF_CACHE_DIR)}",
    "HUGGINGFACE_HUB_CACHE": f"/app/{repo_rel(HF_CACHE_DIR / "hub")}",
    "TRANSFORMERS_CACHE": f"/app/{repo_rel(HF_CACHE_DIR / "transformers")}",
}

print(json.dumps({
    "run_tag": RUN_TAG,
    "split": SPLIT,
    "start_query_idx": START_QUERY_IDX,
    "max_queries": MAX_QUERIES,
    "seed": SEED,
    "run_dir": str(RUN_DIR),
    "runs_json": str(RUNS_JSON),
    "official_json": str(OFFICIAL_JSON),
}, indent=2))


{
  "run_tag": "20260317_112916",
  "split": "validation",
  "start_query_idx": 0,
  "max_queries": null,
  "seed": 42,
  "run_dir": "/Users/lotfi/Documents/EMLV/Memoire/StigmergiAgentic/output/travelplanner_official_full_eval/20260317_112916",
  "runs_json": "/Users/lotfi/Documents/EMLV/Memoire/StigmergiAgentic/output/travelplanner_official_full_eval/20260317_112916/runs.json",
  "official_json": "/Users/lotfi/Documents/EMLV/Memoire/StigmergiAgentic/output/travelplanner_official_full_eval/20260317_112916/official_eval.json"
}


In [9]:
if BUILD_IMAGE:
    run_command(["docker", "compose", "build", "travelplanner-smoke"], log_path=BUILD_LOG, check=True)

if PREPARE_DATA:
    run_command(
        docker_compose_run("python", "scripts/setup_travelplanner.py", env_vars=DOCKER_ENV),
        log_path=DATA_LOG,
        check=True,
    )

count_code = (
    "from datasets import load_dataset\n"
    f"ds = load_dataset('osunlp/TravelPlanner', '{SPLIT}')\n"
    f"print(len(ds['{SPLIT}']))\n"
)
count_result = run_command(
    docker_compose_run("python", "-c", count_code, env_vars=DOCKER_ENV),
    log_path=COUNT_LOG,
    check=True,
)

TOTAL_QUERIES = int(count_result["output"].strip().splitlines()[-1])
END_QUERY_IDX = TOTAL_QUERIES if MAX_QUERIES is None else min(TOTAL_QUERIES, START_QUERY_IDX + MAX_QUERIES)
QUERY_INDICES = list(range(START_QUERY_IDX, END_QUERY_IDX))

print(json.dumps({
    "total_queries_in_split": TOTAL_QUERIES,
    "scheduled_queries": len(QUERY_INDICES),
    "first_query_idx": QUERY_INDICES[0] if QUERY_INDICES else None,
    "last_query_idx": QUERY_INDICES[-1] if QUERY_INDICES else None,
}, indent=2))


$ docker compose build travelplanner-smoke
 Image stigmergiagentic-travelplanner-smoke Building 
#1 [internal] load local bake definitions
#1 reading from stdin 626B done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 1.47kB 0.0s done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/python:3.11-slim
#3 DONE 0.1s

#4 [internal] load .dockerignore
#4 transferring context: 506B done
#4 DONE 0.0s

#5 [builder 1/5] FROM docker.io/library/python:3.11-slim@sha256:c24e9effa2821a6885165d930d939fec2af0dcf819276138f11dd45e200bd032
#5 resolve docker.io/library/python:3.11-slim@sha256:c24e9effa2821a6885165d930d939fec2af0dcf819276138f11dd45e200bd032 0.0s done
#5 DONE 0.0s

#6 [internal] load build context
#6 transferring context: 10.19MB 0.2s done
#6 DONE 0.2s

#7 [builder 3/5] WORKDIR /build
#7 CACHED

#8 [runner 3/5] COPY --from=builder /opt/venv /opt/venv
#8 CACHED

#9 [builder 4/5] COPY requirements.txt .
#9 CACHED

#10 [builder 5/5] 

In [10]:
runs_payload = load_json(
    RUNS_JSON,
    {
        "run_tag": RUN_TAG,
        "split": SPLIT,
        "total_queries": TOTAL_QUERIES,
        "start_query_idx": START_QUERY_IDX,
        "max_queries": MAX_QUERIES,
        "seed": SEED,
        "runs": [],
    },
)
runs = runs_payload.get("runs", []) if isinstance(runs_payload, dict) else []
runs_by_idx = {
    int(run["query_idx"]): run
    for run in runs
    if isinstance(run, dict) and "query_idx" in run
}

for position, query_idx in enumerate(QUERY_INDICES, start=1):
    if query_idx in runs_by_idx:
        print(f"[{position}/{len(QUERY_INDICES)}] skipping query {query_idx} (already checkpointed)")
        continue

    objective = f"Query {query_idx}"
    log_path = QUERY_DIR / f"query_{query_idx:03d}.log"
    query_json = QUERY_DIR / f"query_{query_idx:03d}.json"
    cmd = docker_compose_run(
        "python",
        f"/app/{repo_rel(QUERY_RUNNER)}",
        "--objective", objective,
        "--query-idx", str(query_idx),
        "--seed", str(SEED),
        "--max-ticks", str(MAX_TICKS),
        env_vars=DOCKER_ENV,
    )
    result = run_command(cmd, log_path=log_path, check=False)

    if result["returncode"] == 0:
        try:
            run_payload = extract_tail_json(result["output"])
            run_payload["status"] = "ok"
        except Exception as exc:  # noqa: BLE001
            run_payload = synthetic_failed_run(query_idx, result=result, error=f"json_parse_error: {exc}")
    else:
        run_payload = synthetic_failed_run(query_idx, result=result, error="runtime_failure")

    run_payload["query_idx"] = int(query_idx)
    run_payload["log_path"] = str(log_path)
    save_json(query_json, run_payload)
    runs_by_idx[query_idx] = run_payload

    checkpoint = {
        "run_tag": RUN_TAG,
        "split": SPLIT,
        "total_queries": TOTAL_QUERIES,
        "start_query_idx": START_QUERY_IDX,
        "max_queries": MAX_QUERIES,
        "seed": SEED,
        "runs": [runs_by_idx[idx] for idx in sorted(runs_by_idx)],
    }
    save_json(RUNS_JSON, checkpoint)
    print(f"[{position}/{len(QUERY_INDICES)}] stored query {query_idx} -> {query_json.name}")

    if run_payload.get("status") != "ok" and not CONTINUE_ON_ERROR:
        raise RuntimeError(f"Stopping after query {query_idx}: {run_payload.get("error", "unknown error")}")

runs_payload = load_json(RUNS_JSON, {"runs": []})
print(f"Checkpointed runs: {len(runs_payload.get("runs", []))}")
RUNS_JSON


$ docker compose run --rm -e PYTHONUNBUFFERED=1 -e HF_HOME=/app/output/travelplanner_official_full_eval/20260317_112916/hf_cache -e HUGGINGFACE_HUB_CACHE=/app/output/travelplanner_official_full_eval/20260317_112916/hf_cache/hub -e TRANSFORMERS_CACHE=/app/output/travelplanner_official_full_eval/20260317_112916/hf_cache/transformers travelplanner-smoke python /app/scripts/run_travelplanner_query_export.py --objective 'Query 0' --query-idx 0 --seed 42 --max-ticks 30
 Container stigmergiagentic-travelplanner-smoke-run-652d0a84f40f Creating 
 Container stigmergiagentic-travelplanner-smoke-run-652d0a84f40f Created 
{
  "assistant_response": "Day 1: from Washington to Myrtle Beach | transport=Flight Number: F3927581, from Washington to Myrtle Beach | breakfast=- | attraction=- | lunch=- | dinner=Catfish Charlie's, Myrtle Beach | accommodation=Yellow submarine, Myrtle Beach\nDay 2: Myrtle Beach | transport=- | breakfast=First Eat, Myrtle Beach | attraction=SkyWheel Myrtle Beach, Myrtle Beach |

PosixPath('/Users/lotfi/Documents/EMLV/Memoire/StigmergiAgentic/output/travelplanner_official_full_eval/20260317_112916/runs.json')

In [11]:
official_result = run_command(
    docker_compose_run(
        "python",
        "scripts/eval_travelplanner_official.py",
        "--runs-json", f"/app/{repo_rel(RUNS_JSON)}",
        "--database-root", "/app/data/travelplanner/database",
        "--split", SPLIT,
        "--out", f"/app/{repo_rel(OFFICIAL_JSON)}",
        env_vars=DOCKER_ENV,
    ),
    log_path=OFFICIAL_LOG,
    check=True,
)

official_payload = load_json(OFFICIAL_JSON, {})
print(json.dumps(official_payload.get("scores", {}), indent=2))
OFFICIAL_JSON


$ docker compose run --rm -e PYTHONUNBUFFERED=1 -e HF_HOME=/app/output/travelplanner_official_full_eval/20260317_112916/hf_cache -e HUGGINGFACE_HUB_CACHE=/app/output/travelplanner_official_full_eval/20260317_112916/hf_cache/hub -e TRANSFORMERS_CACHE=/app/output/travelplanner_official_full_eval/20260317_112916/hf_cache/transformers travelplanner-smoke python scripts/eval_travelplanner_official.py --runs-json /app/output/travelplanner_official_full_eval/20260317_112916/runs.json --database-root /app/data/travelplanner/database --split validation --out /app/output/travelplanner_official_full_eval/20260317_112916/official_eval.json
 Container stigmergiagentic-travelplanner-smoke-run-77be078c15af Creating 
 Container stigmergiagentic-travelplanner-smoke-run-77be078c15af Created 
{
  "runs_json": "/app/output/travelplanner_official_full_eval/20260317_112916/runs.json",
  "database_root": "/app/data/travelplanner/database",
  "split": "validation",
  "predicted_queries": [
    0,
    1,
    2

PosixPath('/Users/lotfi/Documents/EMLV/Memoire/StigmergiAgentic/output/travelplanner_official_full_eval/20260317_112916/official_eval.json')

In [12]:
runs_payload = load_json(RUNS_JSON, {"runs": []})
runs = runs_payload.get("runs", [])
official_payload = load_json(OFFICIAL_JSON, {})
scores = official_payload.get("scores", {}) if isinstance(official_payload, dict) else {}

status_counts = Counter(str(run.get("status", "unknown")) for run in runs if isinstance(run, dict))
ok_runs = [run for run in runs if isinstance(run, dict) and run.get("status") == "ok"]
runtime_failures = [
    {
        "query_idx": run.get("query_idx"),
        "error": run.get("error", ""),
        "log_path": run.get("log_path", ""),
    }
    for run in runs
    if isinstance(run, dict) and run.get("status") != "ok"
]

total_cost = sum(float((run.get("summary") or {}).get("cost_used", 0.0)) for run in ok_runs)
total_tokens = sum(int((run.get("summary") or {}).get("tokens_used", 0)) for run in ok_runs)

report = {
    "run_tag": RUN_TAG,
    "split": SPLIT,
    "queries_attempted": len(runs),
    "status_counts": dict(status_counts),
    "total_tokens": total_tokens,
    "total_cost_usd": total_cost,
    "official_scores": scores,
    "runtime_failures": runtime_failures[:10],
    "runs_json": str(RUNS_JSON),
    "official_json": str(OFFICIAL_JSON),
}

print(json.dumps(report, indent=2))


{
  "run_tag": "20260317_112916",
  "split": "validation",
  "queries_attempted": 180,
  "status_counts": {
    "ok": 180
  },
  "total_tokens": 1627374,
  "total_cost_usd": 0.14728555,
  "official_scores": {
    "delivery_rate": 0.5833333333333334,
    "commonsense_micro": 0.45416666666666666,
    "commonsense_macro": 0.17777777777777778,
    "hard_constraint_micro": 0.22857142857142856,
    "hard_constraint_macro": 0.14444444444444443,
    "final_pass_rate": 0.1,
    "evaluated_queries": 180,
    "official_detailed": {
      "Commonsense Constraint": {
        "easy": {
          "3": {
            "Reasonable City Route": {
              "true": 10,
              "false": 10,
              "total": 20
            },
            "Diverse Restaurants": {
              "true": 20,
              "false": 0,
              "total": 20
            },
            "Diverse Attractions": {
              "true": 20,
              "false": 0,
              "total": 20
            },
           